# Step 1: Install Required Libraries

In this step, we install all the necessary libraries required to build the RAG-based Customer Support Assistant.

Libraries:
- langchain → for building LLM pipelines
- chromadb → for vector database
- pypdf → for reading PDF files
- sentence-transformers → for embeddings
- langgraph → for workflow orchestration

In [3]:
!pip install -q langchain langchain-community langchain-openai chromadb pypdf sentence-transformers langgraph requests==2.32.4

# Step 2: Import Required Libraries

In this step, we import all the necessary modules for:
- PDF loading
- Text chunking
- Embedding generation
- Vector storage
- Retrieval
- Workflow (LangGraph)

In [4]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma

from langchain.chains import RetrievalQA
from langchain.llms import OpenAI

from langgraph.graph import StateGraph

# Step 3: Upload and Load PDF

In this step:
- We upload the customer support PDF file
- Load it using PyPDFLoader
- Convert it into documents for further processing

In [5]:
from google.colab import files
uploaded = files.upload()

Saving ecommerce-faqs.pdf to ecommerce-faqs.pdf


In [7]:
loader = PyPDFLoader("ecommerce-faqs.pdf")  # replace with actual file name
documents = loader.load()

print(len(documents))

2


# Step 4: Text Chunking

In this step:
- We split the document into smaller chunks
- This helps in better retrieval accuracy
- Chunking ensures the LLM gets relevant context instead of full document

Strategy:
- Chunk size = 500 characters
- Overlap = 50 characters (to maintain continuity)

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 15


# Step 5: Embeddings and Vector Storage

In this step:
- Convert text chunks into embeddings (numerical vectors)
- Store embeddings in a vector database (ChromaDB)
- This enables similarity-based retrieval

Why embeddings?
- They capture semantic meaning of text
- Help find relevant answers instead of keyword matching

In [9]:
# Create embeddings
embedding = HuggingFaceEmbeddings()

# Store in ChromaDB
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding
)

print("Vector DB created successfully")

/tmp/ipykernel_46543/2905875835.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings()
/tmp/ipykernel_46543/2905875835.py:2: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedding = HuggingFaceEmbeddings()
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://hu

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector DB created successfully


# Step 6: Retriever and Query Testing

In this step:
- We create a retriever from the vector database
- The retriever fetches relevant chunks based on user query
- We then test the system with a sample question

Retriever:
- Uses similarity search
- Returns top-k relevant chunks

In [10]:
# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Test retrieval
query = "What is the refund policy?"

docs = retriever.get_relevant_documents(query)

for i, doc in enumerate(docs):
    print(f"\nResult {i+1}:")
    print(doc.page_content)


Result 1:
OPERATIONAL
What is a MID?
•  MID stands for “Merchant Identification Number” and is used to identify your merchant account.
What are chargebacks and why do they occur?
•  A chargeback is a transaction for which you have been paid but is subsequently disputed by the cardholder 
or issuer. The cardholder can dispute transactions for several reasons, and the card issuer will support the 
cardholder’s dispute if it is valid under the Visa and MasterCard scheme rules.
What is a Virtual Terminal?

Result 2:
What is a recurring transaction and which card types are accepted?
•  Recurring payments enable merchants to accept regular payments on a card, based on the details provided 
in an initial transaction from a merchant website (hosted payment page or API) or through the Virtual Terminal. 
It is an alternative to direct debits and can be accepted for any of the following card types: 
 MasterCard
®
 
 Visa
®
 
 Diners/Discover
®
 
 American Express
®

Result 3:
•  Each user can ch

/tmp/ipykernel_46543/1739053135.py:7: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(query)


# Step 7: LLM Integration and Answer Generation

In this step:
- We connect the retriever with an LLM
- The LLM generates answers using retrieved context
- This completes the RAG pipeline

We are using a lightweight HuggingFace model (GPT-2) for demonstration.

In [19]:
from transformers import pipeline
from langchain.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA

# Create text generation pipeline
pipe = pipeline(
    "text-generation",
    model="gpt2",
    max_new_tokens=60,
    temperature=0.2,
    return_full_text=False
)
# Wrap as LangChain LLM
llm = HuggingFacePipeline(pipeline=pipe)

# Create RAG QA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=False
)

# Test query
query = "What is the refund policy?"

result = qa_chain.invoke(query)

print(result["result"])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




•  The refund policy is a simple one-time payment of the amount of the payment you received.

•  The refund policy is not a refund for any reason.

•  The refund policy is not a refund for any reason.

•  The refund policy


In [20]:
from langchain.prompts import PromptTemplate

prompt_template = """
Answer the question using only the context below.
Give a short and clear answer.

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=False
)

# Step 8: LangGraph Workflow

In this step:
- We create a graph-based workflow using LangGraph
- It controls how the query flows through the system

Flow:
User Input → Processing Node → Output

In [21]:
from typing import TypedDict
from langgraph.graph import StateGraph

# Define state structure
class State(TypedDict):
    query: str
    response: str

# Processing node
def process_node(state):
    result = qa_chain.invoke(state["query"])
    return {"response": result["result"]}

# Create graph
graph = StateGraph(State)

# Add node
graph.add_node("process", process_node)

# Set entry point
graph.set_entry_point("process")

# Compile graph
app = graph.compile()

In [22]:
result = app.invoke({"query": "What is the refund policy?"})
print(result["response"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



•  The refund policy is based on the amount of money that you have paid.

•  The refund policy is based on the amount of money that you have paid.

•  The refund policy is based on the amount of money that you have paid.

• 


# Step 9: Human-in-the-Loop (HITL)

In this step:
- We add escalation logic to the system
- If the model cannot answer properly, it forwards the query to a human agent

Escalation Conditions:
- Low confidence response
- Very short or unclear answer
- Query not found in document

In [23]:
# Updated processing node with HITL logic

def process_node(state):
    result = qa_chain.invoke(state["query"])
    answer = result["result"]

    # Escalation conditions
    if (
        answer is None or
        len(answer.strip()) < 30 or
        "don't know" in answer.lower() or
        "not available" in answer.lower()
    ):
        return {"response": "Escalated to human agent"}

    return {"response": answer}

In [24]:
graph = StateGraph(State)

graph.add_node("process", process_node)

graph.set_entry_point("process")

app = graph.compile()

In [25]:
# Unknown query (should escalate)
result = app.invoke({"query": "What is your CEO salary?"})
print(result["response"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



•  The CEO salary is the total compensation you receive from your company.

•  The CEO salary is the total compensation you receive from your company.

•  The CEO salary is the total compensation you receive from your company.

•  The CEO salary is the total


# High-Level Design (HLD)

## 1. System Overview

### Problem Definition

Customer support systems often fail to provide accurate responses due to lack of context awareness and updated knowledge. Traditional chatbots rely on static rules and cannot retrieve information from documents.

### Solution

A Retrieval-Augmented Generation (RAG)-based system that:

* Processes a PDF knowledge base
* Retrieves relevant information
* Generates contextual answers using LLM
* Uses workflow orchestration via LangGraph
* Supports Human-in-the-Loop (HITL)

---

## 2. Architecture Overview

User → UI → LangGraph → Retriever → LLM → Response / HITL

---

## 3. Components

### Document Loader

Loads PDF using PyPDFLoader

### Chunking Module

Splits text into smaller chunks (500 size, 50 overlap)

### Embedding Model

Uses HuggingFace embeddings to convert text into vectors

### Vector Database

Uses ChromaDB for storing embeddings

### Retriever

Fetches top-k relevant chunks

### LLM

Generates answers using context

### Workflow Engine

LangGraph manages execution flow

### Routing Layer

Decides between answer or escalation

### HITL Module

Handles escalation to human agent

---

## 4. Data Flow

PDF → Text → Chunks → Embeddings → Vector DB
Query → Retriever → Context → LLM → Output

---

## 5. Technology Choices

* ChromaDB → Lightweight and fast
* LangGraph → Workflow control
* HuggingFace → Free LLM

---

## 6. Scalability

* Supports large documents via chunking
* Can handle multiple queries
* Can be extended to multi-document system


# Low-Level Design (LLD)

## 1. Modules

* Document Loader Module
* Chunking Module
* Embedding Module
* Vector Storage Module
* Retrieval Module
* Query Processing Module
* Graph Execution Module
* HITL Module

---

## 2. Data Structures

### Document

* page_content: string
* metadata: dict

### Chunk

* text: string
* id: integer

### Embedding

* vector: float[]

### State (LangGraph)

* query: string
* response: string

---

## 3. Workflow Design

Node: process_node
Input: query
Output: response

Flow: Input → Process → Output

---

## 4. Conditional Logic

Escalation when:

* Answer length < 30
* Contains “don’t know”
* No relevant context

---

## 5. HITL Design

* Triggered when system fails
* Returns “Escalated to human agent”
* Can be extended to real support system

---

## 6. API Design

Input:
{
"query": "string"
}

Output:
{
"response": "string"
}

---

## 7. Error Handling

* Missing PDF
* No chunks found
* LLM failure


# Technical Documentation

## 1. Introduction

RAG combines retrieval and generation to improve LLM accuracy.

---

## 2. Architecture Explanation

The system processes documents, stores embeddings, retrieves relevant data, and generates answers.

---

## 3. Design Decisions

* Chunk size = 500 for balance
* Overlap = 50 for continuity
* Top-k retrieval = 3

---

## 4. Workflow

LangGraph manages flow using nodes and state transitions.

---

## 5. Conditional Logic

Used to detect poor responses and trigger escalation.

---

## 6. HITL

Improves reliability by involving human agents.

---

## 7. Challenges

* GPT-2 produces weak responses
* Trade-off between speed and accuracy

---

## 8. Testing

Sample queries:

* What is refund policy?
* How to contact support?

---

## 9. Future Enhancements

* Better LLM (GPT-4)
* Multi-document support
* Web UI
* Memory system
